# Derivación tensorial **realmente simbólica** para teorías $L(R,\mathcal G)$

Este notebook extiende el motor anterior de $f(R)$ para admitir el invariante de Gauss--Bonnet

$$
\mathcal G
=
R_{abcd}R^{abcd}-4R_{ab}R^{ab}+R^2.
$$

Cada línea importante sigue siendo un **objeto simbólico real** almacenado en `S[...]`.

La regla de trabajo es:

$$
L_{\rm input}(R,\mathcal G)
\longrightarrow
L_R,\ L_{\mathcal G}
\longrightarrow
P^{abcd}
\longrightarrow
P^{ab},\ \mathcal R_{ab}
\longrightarrow
\delta(\sqrt{-g}L)
\longrightarrow
\text{Palatini}
\longrightarrow
\text{dos IBP}
\longrightarrow
E_{ab}.
$$

En particular:

- $\partial\mathcal G/\partial R_{abcd}$ se construye término a término desde $R_{abcd}R^{abcd}$, $R_{ab}R^{ab}$ y $R^2$;
- $P^{abcd}$ se obtiene por la regla de la cadena $L_R\,\partial R/\partial R_{abcd}+L_{\mathcal G}\,\partial\mathcal G/\partial R_{abcd}$;
- las dos integraciones por partes conservan exactamente la misma lógica del notebook anterior;
- el caso $L=R+\alpha\mathcal G$ se reconoce automáticamente como un sector de segundo orden porque $\nabla_aP^{abcd}=0$.

Todo queda disponible para reutilizarse, por ejemplo `S["dGB_dRiemann"]`, `S["P_abcd"]`, `S["ibp1_divergence"]`, `S["E_ab_raw"]`, etc.

In [2]:
import sympy as sp
import numpy as np
from frgb_core import FRGBContext

ctx = FRGBContext()
S = ctx.S

# Símbolos disponibles para definir L_input
R, GB, alpha, beta, gamma, Lambda, mu = ctx.expose_user_symbols()

## Input del usuario

Cambia **solo** `L_input`. El resto del notebook vuelve a calcular toda la cadena.

El símbolo `GB` representa

$$
\mathcal G=R_{abcd}R^{abcd}-4R_{ab}R^{ab}+R^2.
$$

Ejemplos válidos:

```python
L_input = R + alpha*GB - 2*Lambda
L_input = R + alpha*R**2 + beta*GB
L_input = R + alpha*GB**2
L_input = R + alpha*R*GB
L_input = sp.exp(beta*R) + alpha*GB
```

En esta versión se recomiendan expresiones explícitas de SymPy en `R` y `GB`.

In [ ]:
from frgb_stage_geometry import configure_input

# ================================================================
# EDITAR SOLO ESTA LÍNEA
# ================================================================
L_input = alpha+ R*alpha + alpha*GB

configure_input(ctx, L_input)

### Datos escalares calculados desde el input

#### L_input
Objeto reutilizable: `S['L_input']`

#### L_R
Objeto reutilizable: `S['L_R']`

#### L_GB
Objeto reutilizable: `S['L_GB']`

#### L_RR
Objeto reutilizable: `S['L_RR']`

#### L_RGB
Objeto reutilizable: `S['L_RGB']`

#### L_GBGB
Objeto reutilizable: `S['L_GBGB']`

# Etapa 1. Construir $P^{abcd}$ desde $L_{\rm input}$

Primero se obtiene

$$
\frac{\partial R}{\partial R_{abcd}}.
$$

Después se construyen por separado

$$
\frac{\partial (R_{ijkl}R^{ijkl})}{\partial R_{abcd}},
\qquad
\frac{\partial (R_{ij}R^{ij})}{\partial R_{abcd}},
\qquad
\frac{\partial R^2}{\partial R_{abcd}},
$$

y se combinan para obtener

$$
\frac{\partial\mathcal G}{\partial R_{abcd}}.
$$

Solo al final se ensambla

$$
P^{abcd}
=
L_R\frac{\partial R}{\partial R_{abcd}}
+
L_{\mathcal G}\frac{\partial\mathcal G}{\partial R_{abcd}}.
$$

In [13]:
from frgb_stage_geometry import stage_1_build_P

stage_1_build_P(ctx)

#### Coeficiente de R antes de imponer simetrías
Objeto reutilizable: `S['Q_naive']`

#### dR/dRiemann después de antisimetrizar el primer par
Objeto reutilizable: `S['Q_antisym_ab']`

#### dR/dRiemann después de antisimetrizar ambos pares
Objeto reutilizable: `S['Q_antisym_ab_cd']`

#### dR/dRiemann después de simetrizar el intercambio de pares
Objeto reutilizable: `S['dR_dRiemann']`

#### Derivada de Riemann^2 respecto de R_abcd
Objeto reutilizable: `S['P_Riemann2_abcd']`

...

#### Verificación: P^{abcd}+P^{bacd}=0
Objeto: `S['check_P_antisym_ab']`

#### Verificación: P^{abcd}+P^{abdc}=0
Objeto: `S['check_P_antisym_cd']`

#### Verificación: P^{abcd}-P^{cdab}=0
Objeto: `S['check_P_pair_exchange']`

#### Verificación: P_GB^{abcd}+P_GB^{bacd}=0
Objeto: `S['check_P_GB_antisym_ab']`

#### Verificación: P_GB^{abcd}-P_GB^{cdab}=0
Objeto: `S['check_P_GB_pair_exchange']`

# Etapa 2. Calcular $P^{ab}=\left(\partial L/\partial g_{ab}\right)_{R_{ijkl}}$ desde el input

Aquí tampoco se usa $P^{ab}=-2\mathcal R^{ab}$.

Se calculan directamente:

1. $\partial R/\partial g_{ab}$;
2. $\partial(R_{ijkl}R^{ijkl})/\partial g_{ab}$;
3. $\partial(R_{ij}R^{ij})/\partial g_{ab}$;
4. $\partial R^2/\partial g_{ab}$;
5. $\partial\mathcal G/\partial g_{ab}$.

Después se aplica la regla de la cadena del $L_{\rm input}$.


In [14]:
from frgb_stage_geometry import stage_2_metric_derivative

stage_2_metric_derivative(ctx)

#### R escrito como contracción tensorial real
Objeto reutilizable: `S['R_scalar_tensor']`

#### GB escrito a partir de Riemann^2, Ricci^2 y R^2
Objeto reutilizable: `S['GB_scalar_tensor']`

#### Derivada métrica de R antes de canonizar
Objeto reutilizable: `S['dR_dg_cov_raw']`

#### Derivada métrica de R canonizada
Objeto reutilizable: `S['dR_dg_cov']`

#### Derivada métrica de Riemann^2
Objeto reutilizable: `S['dRiemann2_dg_cov']`

#### Derivada métrica de Ricci^2
Objeto reutilizable: `S['dRicci2_dg_cov']`

#### Derivada métrica de R^2
Objeto reutilizable: `S['dR2_dg_cov']`

#### Derivada métrica de GB calculada término a término
Objeto reutilizable: `S['dGB_dg_cov']`

#### P^{ab} completo calculado directamente desde L_input
Objeto reutilizable: `S['P_metric_ab']`

#### Verificación: P^{ab}-P^{ba}=0
Objeto: `S['check_P_metric_symmetry']`

# Etapa 3. Dos cálculos de $\mathcal L_\xi L$, ahora **con el $L$ del usuario**

La primera ruta deriva el escalar $L(R,\mathcal G)$:

$$
\nabla_mL
=
L_R\nabla_mR
+
L_{\mathcal G}\nabla_m\mathcal G.
$$

La segunda ruta calcula por separado

$$
P^{ab}\mathcal L_\xi g_{ab},
\qquad
P^{ijkl}\mathcal L_\xi R_{ijkl},
$$

usando los $P$ ya calculados desde el input.


In [15]:
from frgb_stage_geometry import stage_3_lie_derivative

stage_3_lie_derivative(ctx)

#### ∇_m R calculado desde Riemann
Objeto reutilizable: `S['nabla_R_from_Riemann']`

#### ∇_m GB calculado desde dGB/dRiemann
Objeto reutilizable: `S['nabla_GB_from_Riemann']`

#### ∇_m L por regla de la cadena en R y GB
Objeto reutilizable: `S['nabla_L']`

#### Primera ruta para la derivada de Lie
Objeto reutilizable: `S['Lie_L_route_1']`

#### L_xi g_ab construido tensorialmente
Objeto reutilizable: `S['Lie_metric_ab']`

...

#### Verificación: T_4-T_1=0
Objeto: `S['check_lie_curv_term_4_equals_1']`

#### Suma calculada de los cuatro términos
Objeto reutilizable: `S['lie_curv_four_sum']`

#### P^{ijkl} L_xi R_{ijkl}
Objeto reutilizable: `S['Lie_Riemann_contraction']`

#### Segunda ruta completa
Objeto reutilizable: `S['Lie_L_route_2']`

#### Verificación: Ruta 2 - Ruta 1 = 0
Objeto: `S['check_two_Lie_routes']`


# Etapa 4. Calcular $\mathcal R^{ab}$ y obtener la identidad principal

$\mathcal R^{ab}$ se construye por contracción del $P^{abcd}$ que salió del input. Después se compara con $P^{ab}$, también calculado directamente.


In [16]:
from frgb_stage_geometry import stage_4_Rcal

stage_4_Rcal(ctx)

#### Rcal^{ab} calculado por contracción
Objeto reutilizable: `S['Rcal_up_ab']`

#### Rcal_ab calculado bajando índices
Objeto reutilizable: `S['Rcal_down_ab']`

#### Verificación: P^{ab}+2 Rcal^{ab}=0
Objeto: `S['check_main_identity']`

#### Verificación: Rcal^{ab}-Rcal^{ba}=0
Objeto: `S['check_Rcal_symmetry']`


# Etapa 5. Variar $\sqrt{-g}L$ con objetos simbólicos

Se cambia de $\delta g_{ab}$ a $\delta g^{ab}$ mediante una contracción real. Luego se construye la variación de la densidad.


In [17]:
from frgb_stage_variation import stage_5_vary_density

stage_5_vary_density(ctx)

#### ∂L/∂g^{ab} obtenido por cambio de variable
Objeto reutilizable: `S['P_metric_contravariant_variable_ab']`

#### Parte métrica de δL
Objeto reutilizable: `S['delta_L_metric']`

#### Parte de curvatura de δL
Objeto reutilizable: `S['delta_L_curvature_unsplit']`

#### δL completo antes de Palatini
Objeto reutilizable: `S['delta_L_total_unsplit']`

#### δ√(-g) como objeto tensorial
Objeto reutilizable: `S['delta_sqrt_minus_g']`

#### δ(√(-g)L) antes de separar δR
Objeto reutilizable: `S['delta_density_unsplit']`

                                                      0₀0₁        ⎛GB⋅α⋅sqrt_m ↪
(2⋅R⋅α⋅sqrt_minus_g + α⋅sqrt_minus_g)⋅\delta\mathrm{R}         + -⎜─────────── ↪
                                                          0₀0₁    ⎝        2   ↪

↪ inus_g   R⋅α⋅sqrt_minus_g   α⋅sqrt_minus_g⎞           0₀                     ↪
↪ ────── + ──────────────── + ──────────────⎟⋅\mathrm{H}     + (4⋅R⋅α⋅sqrt_min ↪
↪                 2                 2       ⎠             0₀                   ↪

↪                                    0₀0₁             0₂                       ↪
↪ us_g + 2⋅α⋅sqrt_minus_g)⋅\mathrm{H}    ⋅\mathrm{R}         + -8⋅α⋅sqrt_minus ↪
↪                                                   0₀  0₁0₂                   ↪

↪                    0₀0₁  0₂             0₃                                   ↪
↪ _g⋅\delta\mathrm{R}        ⋅\mathrm{R}         + 2⋅α⋅sqrt_minus_g⋅\delta\mat ↪
↪                        0₀             0₁  0₂0₃                               ↪

↪       0₀0₁0₂0₃        

#### Verificación: ∂L/∂g^{ab} - 2 Rcal_ab = 0
Objeto: `S['check_metric_variation_equals_2Rcal']`


# Etapa 6. Separar $\delta R_{abcd}$

Se sustituye

$R_{abcd}=g_{ae}R^e{}_{bcd}$

y se calcula explícitamente la pieza que contiene $\delta g_{ae}$, ya expresada en términos de $H^{ab}\equiv\delta g^{ab}$.


In [18]:
from frgb_stage_variation import stage_6_split_delta_R

stage_6_split_delta_R(ctx)

#### Primera pieza de P δR, realmente contraída
Objeto reutilizable: `S['delta_R_split_metric_piece']`

#### Segunda pieza de P δR
Objeto reutilizable: `S['delta_R_split_connection_piece']`

#### Verificación: P δg R + Rcal_ab H^ab = 0
Objeto: `S['check_split_metric_piece']`


# Etapa 7. Identidad de Palatini: combinar los dos términos

Se reemplaza $\delta R^e{}_{bcd}$ por dos derivadas de $\delta\Gamma$. El CAS prueba que, después de la antisimetría de $P$, ambos aportes son iguales.


In [19]:
from frgb_stage_variation import stage_7_palatini

stage_7_palatini(ctx)

#### palatini_term_1_raw
Objeto reutilizable: `S['palatini_term_1_raw']`

#### palatini_term_2_raw
Objeto reutilizable: `S['palatini_term_2_raw']`

#### palatini_term_1
Objeto reutilizable: `S['palatini_term_1']`

#### palatini_term_2
Objeto reutilizable: `S['palatini_term_2']`

#### palatini_sum
Objeto reutilizable: `S['palatini_sum']`

#### Verificación: segundo término - primer término = 0
Objeto: `S['check_palatini_two_terms_equal']`

#### Verificación: suma - 2×primer término = 0
Objeto: `S['check_palatini_sum_is_twice']`


# Etapa 8. Sustituir $\delta\Gamma$ y reducir los tres términos

Ahora sí se construye

$\nabla_c\delta\Gamma^e{}_{db}$

a partir de $DDh_{cd\,bi}\equiv\nabla_c\nabla_d\delta g_{bi}$. Los tres términos se guardan y canonizan por separado.


In [20]:
from frgb_stage_variation import stage_8_substitute_dGamma

stage_8_substitute_dGamma(ctx)

#### Pieza 1 antes de canonizar
Objeto reutilizable: `S['dGamma_piece_1_raw']`

#### Pieza 1 canonizada
Objeto reutilizable: `S['dGamma_piece_1']`

#### Pieza 2 antes de canonizar
Objeto reutilizable: `S['dGamma_piece_2_raw']`

#### Pieza 2 canonizada
Objeto reutilizable: `S['dGamma_piece_2']`

#### Pieza 3 antes de canonizar
Objeto reutilizable: `S['dGamma_piece_3_raw']`

...

#### after_dGamma_full
Objeto reutilizable: `S['after_dGamma_full']`

#### Verificación: primera pieza = 0 por antisimetría/simetría
Objeto: `S['check_dGamma_piece_1_vanishes']`

#### Verificación: pieza 2 - pieza 3 = 0
Objeto: `S['check_dGamma_piece_2_equals_3']`

#### Verificación: expansión completa - combinación reducida = 0
Objeto: `S['check_after_dGamma_reduction']`

#### Resultado calculado tras sustituir δΓ
Objeto reutilizable: `S['palatini_metric_second_derivative']`


# Etapa 9. Primera integración por partes, verificada por regla del producto

No se escribe una igualdad decorativa. Se construye el vector de borde $B_1^j$, se calcula su divergencia por regla del producto y se verifica algebraicamente la identidad.


In [21]:
from frgb_stage_ibp import stage_9_ibp_first

stage_9_ibp_first(ctx)

#### ibp_start
Objeto reutilizable: `S['ibp_start']`

#### ibp1_boundary_vector
Objeto reutilizable: `S['ibp1_boundary_vector']`

#### ibp1_divergence
Objeto reutilizable: `S['ibp1_divergence']`

#### ibp1_residual_positive
Objeto reutilizable: `S['ibp1_residual_positive']`

#### Verificación: integrando inicial - (divergencia - residuo) = 0
Objeto: `S['check_ibp1']`


# Etapa 10. Segunda integración por partes, nuevamente calculada

Se renombra el residuo de la primera IBP, se construye $B_2^j$, se calcula $\nabla_jB_2^j$ y se verifica la segunda identidad.


In [22]:
from frgb_stage_ibp import stage_10_ibp_second

stage_10_ibp_second(ctx)

#### Verificación: residuo renombrado + residuo positivo anterior = 0
Objeto: `S['check_residual_renaming']`

#### ibp1_residual_negative_renamed
Objeto reutilizable: `S['ibp1_residual_negative_renamed']`

#### ibp2_boundary_vector
Objeto reutilizable: `S['ibp2_boundary_vector']`

#### ibp2_divergence
Objeto reutilizable: `S['ibp2_divergence']`

#### ibp2_bulk_hcov
Objeto reutilizable: `S['ibp2_bulk_hcov']`

#### Verificación: residuo - (-divergencia + nuevo bulk) = 0
Objeto: `S['check_ibp2']`


# Etapa 11. Término de borde completo y conversión a $\delta g^{ab}$

El término de borde es la combinación que salió de las dos reglas del producto. No se introduce desde una fórmula final.


In [23]:
from frgb_stage_ibp import stage_11_boundary_and_bulk

stage_11_boundary_and_bulk(ctx)

#### δv^j especializado al L_input
Objeto reutilizable: `S['delta_v_vector']`

#### Bulk después de las dos IBP, en función de δg^{ab}
Objeto reutilizable: `S['ibp2_bulk_Hup']`

#### -2 ∇^m∇^n P_amnb calculado desde las componentes de P
Objeto reutilizable: `S['minus2_double_divergence_P_ab']`

#### Verificación: bulk de IBP - (-2∇∇P)_ab H^ab = 0
Objeto: `S['check_ibp_bulk_equals_double_divergence']`


# Etapa 12. Ensamblar $E_{ab}$ usando exclusivamente piezas ya calculadas

Se suman:

1. $\mathcal R_{ab}$, obtenido de $P\cdot R$;
2. la variación del volumen;
3. el bulk que salió de las dos integraciones por partes.


In [24]:
from frgb_stage_field import stage_12_field_tensor

stage_12_field_tensor(ctx)

#### Tensor de campo construido paso a paso
Objeto reutilizable: `S['E_ab_raw']`

#### Integrando bulk final de δA
Objeto reutilizable: `S['delta_action_bulk_integrand']`

 ⎛GB⋅α⋅sqrt_minus_g   R⋅α⋅sqrt_minus_g   α⋅sqrt_minus_g⎞           0₀          ↪
-⎜───────────────── + ──────────────── + ──────────────⎟⋅\mathrm{H}     + (2⋅R ↪
 ⎝        2                  2                 2       ⎠             0₀        ↪

↪                                             0₀0₁             0₂              ↪
↪ ⋅α⋅sqrt_minus_g + α⋅sqrt_minus_g)⋅\mathrm{H}    ⋅\mathrm{R}         + -4⋅α⋅s ↪
↪                                                            0₀  0₁0₂          ↪

↪                       0₀0₁             0₂  0₃             0₄                 ↪
↪ qrt_minus_g⋅\mathrm{H}    ⋅\mathrm{R}        ⋅\mathrm{R}         + 2⋅α⋅sqrt_ ↪
↪                                      0₀  0₁             0₂  0₃0₄             ↪

↪                   0₀0₁             0₂0₃0₄                                    ↪
↪ minus_g⋅\mathrm{H}    ⋅\mathrm{R}        ⋅\mathrm{R}         + 4⋅α⋅sqrt_minu ↪
↪                                  0₀                 0₁0₂0₃0₄                 ↪

↪               0₀0₁    

#### Contribución derivativa asociada a L_R
Objeto reutilizable: `S['derivative_piece_R_component']`

#### Contribución derivativa asociada a L_GB
Objeto reutilizable: `S['derivative_piece_GB_component']`

#### Verificación: (-2∇∇P)_ab - suma de las contribuciones R y GB = 0
Objeto: `S['check_double_divergence_chain_rule']`

# Etapa 13. Diagnóstico de orden desde $\nabla_aP^{abcd}$

El notebook calcula por separado

$$
\nabla_aL_R,
\qquad
\nabla_aL_{\mathcal G},
$$

y luego construye $\nabla_aP^{abcd}$.

Así distingue automáticamente entre:

- combinaciones lineales de Lovelock, como $R+\alpha\mathcal G$, para las que $\nabla_aP^{abcd}=0$;
- dependencias no lineales en $R$ o $\mathcal G$, que generan términos de orden superior.


In [25]:
from frgb_stage_field import stage_13_order_diagnostic

stage_13_order_diagnostic(ctx)

#### ∇_a L_R
Objeto reutilizable: `S['gradient_L_R']`

#### ∇_a L_GB
Objeto reutilizable: `S['gradient_L_GB']`

#### ∇_a P^{abcd} calculado
Objeto reutilizable: `S['divergence_P_bcd']`

### Diagnóstico automático

**Sector de segundo orden:** los coeficientes de las estructuras $\partial R/\partial R_{abcd}$ y $\partial\mathcal G/\partial R_{abcd}$ son constantes en el espacio-tiempo, por lo que `S['divergence_P_bcd']` se anula identitariamente.

Para un término lineal $lpha\mathcal G$, esta es precisamente la propiedad de Lovelock. En $D=4$, además, el término Gauss--Bonnet lineal es topológico; el motor tensorial abstracto no fija una dimensión concreta, así que esa cancelación específica de $D=4$ no se impone automáticamente.

# Cómo reutilizar cualquier paso

No hay que volver a deducir nada ni copiar LaTeX. Los objetos están en `S`.

Ejemplos:

```python
S["dGB_dRiemann"]
S["P_abcd"]
S["P_metric_ab"]
S["lie_curv_term_3"]
S["after_dGamma_full"]
S["ibp1_divergence"]
S["delta_v_vector"]
S["minus2_double_divergence_P_ab"]
S["E_ab_raw"]
```

Puedes hacer nuevas operaciones:

```python
ctx.tsimplify(S["lie_curv_term_4"] - S["lie_curv_term_1"])
ctx.tsimplify(S["E_ab_raw"])
sp.diff(S["L_input"], GB, 2)
```


In [26]:
from frgb_stage_field import show_stored_objects

show_stored_objects(ctx)

### Se almacenaron 117 objetos simbólicos

L_input
L_R
L_GB
L_RR
L_RGB
L_GBGB
Q_naive
Q_antisym_ab
Q_antisym_ab_cd
dR_dRiemann
P_Riemann2_term_1
P_Riemann2_term_2
P_Riemann2_abcd
P_Ricci2_raw
P_Ricci2_antisym_ab
P_Ricci2_antisym_ab_cd
P_Ricci2_abcd
P_R2_abcd
dGB_dRiemann
P_abcd
check_P_antisym_ab
check_P_antisym_cd
check_P_pair_exchange
check_P_GB_antisym_ab
check_P_GB_pair_exchange
R_scalar_tensor
dR_dg_cov_raw
dR_dg_cov
Riemann2_scalar_tensor
dRiemann2_dg_cov
Ricci2_scalar_tensor
dRicci2_dg_cov
dR2_dg_cov
GB_scalar_tensor
dGB_dg_cov
P_metric_ab
P_metric_template_indices
check_P_metric_symmetry
nabla_R_from_Riemann
nabla_GB_from_Riemann
nabla_L
Lie_L_route_1
Lie_metric_ab
Lie_metric_contraction
lie_curv_transport
lie_curv_term_1_raw
lie_curv_term_1
lie_curv_term_2_raw
lie_curv_term_2
lie_curv_term_3_raw
lie_curv_term_3
lie_curv_term_4_raw
lie_curv_term_4
check_lie_curv_term_2_equals_1
check_lie_curv_term_3_equals_1
check_lie_curv_term_4_equals_1
lie_curv_four_sum
Lie_Riemann_contraction
Lie_L_route_2
check_two_Lie_routes
Rcal_

# Etapa 14. Exportar las expresiones principales

Este apartado **no recalcula** la derivación. Toma directamente los objetos ya obtenidos y almacenados en `S` para:

1. mostrar una vista compacta de las expresiones principales;
2. guardar un archivo `.tex` independiente y listo para compilar;
3. guardar un fragmento `.tex` reutilizable;
4. compilar automáticamente un PDF cuando `pdflatex` esté disponible;
5. añadir una tabla final con todas las verificaciones `check_*`.

Así, el resumen final siempre depende del `L_input` usado en la ejecución.


In [27]:
from frgb_export import export_results

_export_info = export_results(
    ctx,
    mostrar_vista_previa=True,
    compilar_pdf=True,
)

## Vista previa compacta

### Entrada y derivadas escalares

**`S['L_input']`**

\[
L = \alpha \left(\mathcal{G} + R + 1\right)
\]

**`S['L_R']`**

\[
L_R = \alpha
\]

**`S['L_GB']`**

\[
L_{\mathcal G} = \alpha
\]

...

## Archivos generados

o:\Mi unidad\Física - PUCP\2026-1\Relatividad General\investigación\CalculosTipo_f(R)_GB\salidas_FR_GB\resumen_derivacion_FR_GB.tex

o:\Mi unidad\Física - PUCP\2026-1\Relatividad General\investigación\CalculosTipo_f(R)_GB\salidas_FR_GB\ecuaciones_principales_FR_GB.tex

**El PDF no se compiló automáticamente.** El `.tex` quedó guardado y puede compilarse en Overleaf o con `pdflatex`.

No se encontró pdflatex. Los archivos .tex sí fueron generados.
Verificaciones automáticas: 24
Todas dieron cero: True
Carpeta de salida: o:\Mi unidad\Física - PUCP\2026-1\Relatividad General\investigación\CalculosTipo_f(R)_GB\salidas_FR_GB
